<a href="https://colab.research.google.com/github/vera2005/MLCV_course1/blob/main/Lesson4/HW4_additional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание № 4 (level up)

#**Многоклассовая классификация изображений на датасете CIFAR-100**

Суть задачи: построение модели для решения задачи многоклассовой классификации

Чуть подробнее: Вам предлагается поработать с датасетом CIFAR-100, содержащим изображения 100 разных классов. Вам предстоит выбрать 10 (или N) классов, которые вас наиболее заинтересуют, выполнить предобработку датасета и построить модель для классификации выбранных Вами классов.

**0. Импорт библиотек**

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

**1. Загрузка датасета**

Выполняется 1 раз, но очень долго

*Задание*: загрузить тренировочную и тестовые выборки с помощью функции

In [ ]:
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.cifar100.load_data()


Сохранение датасета в среду Коллаб, чтобы не пришлось качать его каждый раз заново

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import os

# Создаём папку для датасета на Google Drive (если её ещё нет)
drive_path = '/content/drive/My Drive/colab_datasets/cifar100_subset'
os.makedirs(drive_path, exist_ok=True)

# Сохраняем все массивы
np.save(f'{drive_path}/x_train.npy', x_train_full)
np.save(f'{drive_path}/y_train.npy', y_train_full)
np.save(f'{drive_path}/x_test.npy', x_test_full)
np.save(f'{drive_path}/y_test.npy', y_test_full)

print('Датасет сохранён в:', drive_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Теперь, чтобы начать работу с датасетом из всего шага 1 достаточно запускать ячейки с этой и ниже)

In [ ]:
import numpy as np

drive_path = '/content/drive/My Drive/colab_datasets/cifar100_subset'

# Загружаем сохранённые массивы
x_train = np.load(f'{drive_path}/x_train.npy')
y_train = np.load(f'{drive_path}/y_train.npy')
x_test  = np.load(f'{drive_path}/x_test.npy')
y_test  = np.load(f'{drive_path}/y_test.npy')

print('Обучающая выборка:', x_train.shape)
print('Тестовая выборка:  ', x_test.shape)

**2. Выбор personallity классов**

Ниже приведены названия всех классов, которые есть в датасете

In [ ]:
# Все 100 названий классов в исходном порядке
all_class_names = ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle',
                   'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel',
                   'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock',
                   'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur',
                   'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster',
                   'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion',
                   'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain',
                   'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree',
                   'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine',
                   'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea',
                   'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider',
                   'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank',
                   'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip',
                   'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman', 'worm']



Ниже код, который поможет вам понять, кем вы хотите стать, когда вырастите (к сожалению, неть...) но поможет выбрать классы. в переменную target_class_name вы можете написать любое название из all_class_names и посмотреть 3 примера, чтобы понять, хотите ли вы классифицировать этих фантастических тварей)

In [ ]:
target_class_name = ''   # можно поменять на любой из chosen_names


# Находим индекс класса в нашем списке (0..9)
target_idx = all_class_names.index(target_class_name)

# Ищем индексы всех картинок этого класса в обучающей выборке
indices = np.where(y_train_full == target_idx)[0]

# Берём первые 3
n_show = min(3, len(indices))
sample_indices = indices[:n_show]

# Рисуем
plt.figure(figsize=(3 * n_show, 4))
for i, idx in enumerate(sample_indices):
    plt.subplot(1, n_show, i + 1)
    plt.imshow(x_train_full[idx])
    plt.title(f'{target_class_name}\n(index={idx}, label={y_train_full[idx]})')
    plt.axis('off')

plt.suptitle(f'3 примера класса "{target_class_name}"', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Всего изображений класса "{target_class_name}" в обучающей выборке: {len(indices)}')

Выбор 10 (или N) интересных для Вас классов

In [ ]:
chosen_names = []

Выбираем из общего датасета представителей Ваших классов

In [ ]:
# Индексы этих классов в исходном датасете
chosen_indices = [all_class_names.index(name) for name in chosen_names]
print("Индексы выбранных классов:", chosen_indices)

# Оставляем только изображения выбранных классов (для train и test)
def filter_by_classes(x, y, chosen_indices):
    # y имеет форму (N, 1), сжимаем до (N,)
    y_flat = y.squeeze()
    # Маска: True, если метка входит в число выбранных
    mask = np.isin(y_flat, chosen_indices)
    x_filtered = x[mask]
    y_filtered = y_flat[mask]
    # Переиндексируем метки: старые индексы -> 0..9
    remap = {old: new for new, old in enumerate(chosen_indices)}
    y_filtered = np.array([remap[label] for label in y_filtered])
    return x_filtered, y_filtered

x_train, y_train = filter_by_classes(x_train_full, y_train_full, chosen_indices)
x_test, y_test   = filter_by_classes(x_test_full,  y_test_full,  chosen_indices)

print("Обучающая выборка:", x_train.shape, "| метки:", y_train.shape)
print("Тестовая выборка:  ", x_test.shape,  "| метки:", y_test.shape)

**3. Нормализация**

Добавлять доп канал нам не нужно, т.к. изображения цветные, то их форма сразу (32, 32, 3)

А вот нормализовать эти тензоры, в которых хранятся изображения, придется.  

Приводим числа из диапазона 0-255 в 0-1

In [ ]:
x_train =
x_test  =

class_names = chosen_names  # для удобства


**4. Визуализация примеров из обучающей выборки**

In [ ]:
plt.figure(figsize=(12, 5))
for i in range(len(class_names)):
    plt.subplot(2, len(class_names)//2, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.suptitle('Примеры из обучающей выборки (CIFAR-100-subset)')
plt.tight_layout()
plt.show()

**5. Конструирование CNN**

Как обычно создаем переменную model и складываем в Sequential слои, которые хотим.

Совет: первый слой - Input(shape=(укажите 3 параметра формы картинки))

Далее используйте изученные слои (свертки, пулинги, дропауты, полносвязные)

не забывайте и о регуляризации)

**6. Компиляция**

Параметры компиляции и обучения Вы вольны выбрать согласно Вашему творческому чутью. Ваша задача получить наивысший результат материк на тестовой выборке)

Творите!)

**7. Обучение**

Совет: можете попробовать воспользоваться объектом EarlyStopping как коллбеком, чтобы останавливать переобучение путем отслеживания динамики изменения функции потерь на тренировочной и валидационных выборках



**8. Оценка на тестовой выборке**

С помощью метода evaluate рассчитайте значение функции потерь и точности на тестовой выборке

**9. Гадание (Предсказываем ответы для тестовой выборки)**
Предскажите ответы для объектов тестовой выборки с помощью метода predict.

Внутри y_pred_probs - массив вероятностей формы (K, 10). K - число объектов в тестовой выборке.
Каждая строка — это одна картинка. В строке 10 чисел — вероятности принадлежности к каждому из 10 классов. Сумма чисел в строке всегда = 1 (потому что последний слой модели — Dense(10, activation='softmax')

np.argmax: функция из NumPy, которая возвращает индекс максимального элемента, а не само значение.

axis=1 означает «искать максимум внутри каждой строки, а не по всему массиву сразу

Итог: y_pred — массив целых чисел от 0 до 9 формы (K,). Это финальные предсказания модели: для каждой картинки — какой класс она выбрала.

In [ ]:
y_pred = np.argmax(y_pred_probs, axis=1)


**10. Precision, Recall, F1 (классификационный отчёт)**

Функция classification_report берёт истинные метки и предсказанные метки и сравнивает их по каждому классу отдельно. Также учтите параметры target_names и zero_division

**11. Графики обучения**

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='обучение')
plt.plot(history.history['val_accuracy'], label='валидация')
plt.xlabel('Эпоха'); plt.ylabel('Точность')
plt.legend(); plt.title('Точность')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='обучение')
plt.plot(history.history['val_loss'], label='валидация')
plt.xlabel('Эпоха'); plt.ylabel('Потери')
plt.legend(); plt.title('Потери')

plt.tight_layout()
plt.show()

**12.Матрица ошибок**

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Предсказанный класс')
plt.ylabel('Истинный класс')
plt.title('Матрица ошибок')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Проанализируйте матрицу ошибок и ответьте на вопрос: фоточками объектов каких классов вы бы расширили датасет? Почему?)

**13. Демонстрация предсказаний**

С помощью метода predict сделайте предсказания для 10 объектов из тестовой выборки

С помощью np.argmax определите класс-победитель для каждой картинки, не забудьте, что каждая строчка должна обрабатываться отдельно. Результат сложите в переменную preds

В переменную confs с помощью np.max сложите значение вероятности предсказания, т.е. для каждой строки требуется сохранить максимальное значение из нее


In [ ]:
probs = model.predict(x_test[:10])
preds = np.argmax(probs, axis=1)
confs = np.max(probs, axis=1)

In [ ]:


plt.figure(figsize=(14, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i])
    true_label = class_names[y_test[i]]
    pred_label = class_names[preds[i]]
    color = 'green' if preds[i] == y_test[i] else 'red'
    plt.title(f'Ист: {true_label}\nПред: {pred_label}\n({confs[i]*100:.1f}%)', color=color)
    plt.axis('off')
plt.tight_layout()
plt.show()